# QA 4 (Unit 4): Clustering Implementation and Visualization
### K-Means and DBSCAN on Customer Data

**Objective:** Apply and compare K-Means and DBSCAN clustering algorithms on a synthetic customer dataset, followed by 2D visualization and analysis.


## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## Step 2: Generate Synthetic Customer Dataset

We simulate a customer dataset with features like **Annual Income** and **Spending Score**, similar to the well-known Mall Customer dataset.

In [ ]:
np.random.seed(42)

# Generate 3 natural clusters of customers
X, y_true = make_blobs(
    n_samples=300,
    centers=[
        [30, 20],   # Low income, low spending
        [60, 60],   # Mid income, mid spending
        [85, 85],   # High income, high spending
    ],
    cluster_std=[6, 7, 6],
    random_state=42
)

# Add a few noise/outlier points
noise = np.array([[10, 90], [95, 10], [50, 5], [5, 50], [100, 50]])
X = np.vstack([X, noise])

# Create DataFrame
df = pd.DataFrame(X, columns=['Annual_Income_k$', 'Spending_Score'])

print('Dataset shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## Step 3: Exploratory Data Analysis

In [ ]:
print('Statistical Summary:')
print(df.describe())

plt.figure(figsize=(6, 4))
plt.scatter(df['Annual_Income_k$'], df['Spending_Score'], alpha=0.6, color='steelblue', edgecolors='white', linewidth=0.5)
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('Customer Data — Raw (No Labels)')
plt.tight_layout()
plt.savefig('raw_data.png', dpi=120, bbox_inches='tight')
plt.show()
print('Raw scatter plot saved.')

## Step 4: Data Preprocessing — Standardization

Both K-Means and DBSCAN are distance-based algorithms. Standardizing features ensures no single feature dominates due to scale.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

print('Before scaling — Mean:', df.mean().values.round(2), '| Std:', df.std().values.round(2))
print('After  scaling — Mean:', X_scaled.mean(axis=0).round(4), '| Std:', X_scaled.std(axis=0).round(4))

## Step 5: K-Means Clustering

**K-Means** partitions data into K clusters by minimizing within-cluster variance (inertia). We use the **Elbow Method** to find the optimal K.

In [ ]:
# --- Elbow Method ---
inertias = []
sil_scores = []
K_range = range(2, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(K_range, inertias, 'bo-', linewidth=2)
ax1.axvline(x=3, color='red', linestyle='--', label='Elbow at K=3')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method')
ax1.legend()

ax2.plot(K_range, sil_scores, 'go-', linewidth=2)
ax2.axvline(x=3, color='red', linestyle='--', label='Best K=3')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Scores')
ax2.legend()

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Best K by Silhouette: {K_range[sil_scores.index(max(sil_scores))]}')

In [ ]:
# --- Apply K-Means with K=3 ---
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

df['KMeans_Cluster'] = kmeans_labels

# 2D Visualization
colors = ['#E74C3C', '#2ECC71', '#3498DB']
plt.figure(figsize=(8, 6))

for i, color in enumerate(colors):
    mask = kmeans_labels == i
    plt.scatter(X_scaled[mask, 0], X_scaled[mask, 1],
                c=color, label=f'Cluster {i}', alpha=0.7, edgecolors='white', linewidth=0.5, s=60)

# Plot centroids
centroids = kmeans.cluster_centers_
plt.scatter(centroids[:, 0], centroids[:, 1],
            c='black', marker='X', s=200, zorder=5, label='Centroids')

plt.xlabel('Annual Income (scaled)')
plt.ylabel('Spending Score (scaled)')
plt.title('K-Means Clustering (K=3)')
plt.legend()
plt.tight_layout()
plt.savefig('kmeans_clusters.png', dpi=120, bbox_inches='tight')
plt.show()

print('K-Means Silhouette Score:', round(silhouette_score(X_scaled, kmeans_labels), 4))
print('Cluster sizes:', dict(zip(*np.unique(kmeans_labels, return_counts=True))))

## Step 6: DBSCAN Clustering

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) groups together points that are closely packed and marks points in low-density regions as outliers (label = -1).

**Key Parameters:**
- `eps`: The maximum distance between two samples to be considered neighbors
- `min_samples`: The minimum number of samples in a neighborhood to form a core point

In [ ]:
# --- k-distance plot to find optimal eps ---
from sklearn.neighbors import NearestNeighbors

nbrs = NearestNeighbors(n_neighbors=5).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
distances = np.sort(distances[:, 4])  # 5th nearest neighbor

plt.figure(figsize=(7, 4))
plt.plot(distances, color='purple', linewidth=2)
plt.axhline(y=0.5, color='red', linestyle='--', label='eps ≈ 0.5')
plt.xlabel('Points (sorted by distance)')
plt.ylabel('5th Nearest Neighbor Distance')
plt.title('k-Distance Graph (for eps selection)')
plt.legend()
plt.tight_layout()
plt.savefig('kdistance.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- Apply DBSCAN ---
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

df['DBSCAN_Cluster'] = dbscan_labels

n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print(f'Number of clusters found: {n_clusters}')
print(f'Number of noise points: {n_noise}')
print(f'Cluster distribution: {dict(zip(*np.unique(dbscan_labels, return_counts=True)))}')

# 2D Visualization
unique_labels = sorted(set(dbscan_labels))
palette = ['#E74C3C', '#2ECC71', '#3498DB', '#F39C12', '#9B59B6']
color_map = {-1: '#AAAAAA'}  # Noise = grey
for i, lbl in enumerate([l for l in unique_labels if l != -1]):
    color_map[lbl] = palette[i % len(palette)]

plt.figure(figsize=(8, 6))
for label in unique_labels:
    mask = dbscan_labels == label
    lname = f'Cluster {label}' if label != -1 else 'Noise'
    marker = 'x' if label == -1 else 'o'
    size = 40 if label == -1 else 60
    plt.scatter(X_scaled[mask, 0], X_scaled[mask, 1],
                c=color_map[label], label=lname, alpha=0.8,
                edgecolors='white' if label != -1 else 'none',
                linewidth=0.5, s=size, marker=marker)

plt.xlabel('Annual Income (scaled)')
plt.ylabel('Spending Score (scaled)')
plt.title(f'DBSCAN Clustering (eps=0.5, min_samples=5)')
plt.legend()
plt.tight_layout()
plt.savefig('dbscan_clusters.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 7: Side-by-Side Comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# K-Means
for i, color in enumerate(colors):
    mask = kmeans_labels == i
    ax1.scatter(X_scaled[mask, 0], X_scaled[mask, 1],
                c=color, label=f'Cluster {i}', alpha=0.7, edgecolors='white', linewidth=0.4, s=55)
ax1.scatter(centroids[:, 0], centroids[:, 1], c='black', marker='X', s=200, zorder=5, label='Centroids')
ax1.set_title('K-Means (K=3)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Income (scaled)')
ax1.set_ylabel('Spending Score (scaled)')
ax1.legend()

# DBSCAN
for label in sorted(set(dbscan_labels)):
    mask = dbscan_labels == label
    lname = f'Cluster {label}' if label != -1 else 'Noise'
    marker = 'x' if label == -1 else 'o'
    ax2.scatter(X_scaled[mask, 0], X_scaled[mask, 1],
                c=color_map[label], label=lname, alpha=0.8,
                edgecolors='white' if label != -1 else 'none',
                linewidth=0.4, s=55, marker=marker)
ax2.set_title('DBSCAN (eps=0.5, min_samples=5)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Income (scaled)')
ax2.set_ylabel('Spending Score (scaled)')
ax2.legend()

plt.suptitle('K-Means vs DBSCAN — Customer Segmentation', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('comparison_plot.png', dpi=120, bbox_inches='tight')
plt.show()
print('Side-by-side comparison saved.')

## Step 8: Evaluation Metrics

In [ ]:
# Silhouette Score (only for non-noise points in DBSCAN)
km_sil = silhouette_score(X_scaled, kmeans_labels)

# DBSCAN: exclude noise for silhouette calculation
valid_mask = dbscan_labels != -1
if len(set(dbscan_labels[valid_mask])) > 1:
    db_sil = silhouette_score(X_scaled[valid_mask], dbscan_labels[valid_mask])
else:
    db_sil = None

print('=' * 50)
print('          EVALUATION SUMMARY')
print('=' * 50)
print(f'K-Means Silhouette Score :  {km_sil:.4f}')
print(f'DBSCAN  Silhouette Score :  {db_sil:.4f}' if db_sil else 'DBSCAN: only 1 cluster, N/A')
print(f'K-Means Inertia          :  {kmeans.inertia_:.2f}')
print(f'DBSCAN Clusters Found    :  {n_clusters}')
print(f'DBSCAN Noise Points      :  {n_noise} ({100*n_noise/len(dbscan_labels):.1f}%)')
print('=' * 50)

## Step 9: Analysis

### Difference Between K-Means and DBSCAN Results

| Feature | K-Means | DBSCAN |
|---|---|---|
| Requires K upfront | ✅ Yes (K=3) | ❌ No |
| Detects outliers | ❌ No (forces assignment) | ✅ Yes (labels them -1) |
| Assumes cluster shape | Spherical / convex | Arbitrary shape |
| Handles noise | ❌ All points assigned | ✅ Marks noisy points |
| Key parameter | n_clusters | eps, min_samples |

### Which Performed Better?

For this dataset (blob-shaped clusters + outliers):

- **K-Means** cleanly separates the 3 natural blobs and achieves a high silhouette score. However, it forces the 5 outlier points into the nearest cluster, distorting centroids slightly.
- **DBSCAN** correctly identifies the core clusters AND flags the outlier points as noise (-1), which is a more honest representation of the data structure.

**Conclusion:** DBSCAN is better for **noisy, real-world data** where outlier detection matters. K-Means is better when cluster count is known and data is relatively clean.

### Observations
1. **Cluster 0** (low income, low spending) — budget-conscious customers
2. **Cluster 1** (mid income, mid spending) — mainstream customers  
3. **Cluster 2** (high income, high spending) — premium customers
4. DBSCAN flagged 5 anomalous customers who don't fit any natural group — useful for fraud detection or data quality checks

In [ ]:
# Final summary table
summary = pd.DataFrame({
    'Metric': ['Algorithm', 'Clusters Found', 'Noise/Outliers', 'Silhouette Score', 'Inertia'],
    'K-Means': ['K-Means', '3', 'None (all assigned)', f'{km_sil:.4f}', f'{kmeans.inertia_:.2f}'],
    'DBSCAN':  ['DBSCAN',  str(n_clusters), str(n_noise), f'{db_sil:.4f}' if db_sil else 'N/A', 'N/A']
})
print(summary.to_string(index=False))